# European Company Financials 2019–2025: starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/european_company_financials_sample.csv', recursive=True) or ['european_company_financials_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df.groupby(['country', 'fiscal_year']).size().unstack(fill_value=0)

In [ ]:
m = df[(df.revenue > 0) & df.net_income.notna()].assign(margin=lambda d: (d.net_income / d.revenue).clip(-5, 5))
m.groupby('country').margin.median().sort_values().plot.barh(figsize=(7, 3), title='Median net margin by country (sample)')
m.groupby('country').margin.describe()

In [ ]:
df[df.fiscal_year == 2024].nlargest(20, 'revenue_eur')[['legal_name', 'country', 'revenue_eur', 'net_income', 'currency', 'employees']]

## Get the full record from the Radar API

The sample keeps five headline figures. The API returns the full income statement, balance sheet and cash flow for each period. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/financials', headers={'Authorization': f'Bearer {key}'},
                      json={'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp